In [1]:
import os
import csv
import calendar
import subprocess

import re
import pandas as pd
from Bio import SeqIO

First we will parse genbank file, filter sequences with ambiguous characters

In [4]:
recs = list(SeqIO.parse("raw/Bastrovirus_22102026.gb", "genbank"))

#list with ambiguous nucleotides
ambig_nt = ['n',  # A, T, G, C 
            'r', # Purine (A or G)
            'y', # Pyrimidine (T or C)
            'k', # Keto (G or T)
            'm', # Amino (A or C)
            's', # Strong interaction (3 H bonds) (G or C)
            'w', # Weak interaction (2 H bonds) (A or T)
            'b', # Not A (C or G or T)
            'd', # Not C (A or G or T)
            'h', # Not G (A or C or T)
            'v' # Not T or U (A or C or G)
            ]


recs_noamb = []

for rec in recs:
    total_amb = 0
    for nt in ambig_nt:
        total_amb += rec.seq.lower().count(nt)
    if total_amb/len(rec.seq)< 0.01:
        recs_noamb.append(rec)
    else:
        print(f"{rec.name} has {total_amb} ({round(100*total_amb/len(rec.seq),2)}%) ambiguous nucleotides")
print(f"There are {len(recs_noamb)} sequences with <1% ambiguous nucleotides")

#print(f"There are {len(not_in_meta)} absent in Astroviridae metadata")


SeqIO.write(recs_noamb, "raw/Bastrovirus_22102026_lessamb.gb", "genbank")
SeqIO.write(recs_noamb, "raw/Bastrovirus_22102026_lessamb.fna", "fasta")

OQ835730 has 413 (7.46%) ambiguous nucleotides
OQ802730 has 132 (2.2%) ambiguous nucleotides
OM953852 has 173 (3.01%) ambiguous nucleotides
There are 67 sequences with <1% ambiguous nucleotides


67

In [6]:
def standardize_date(date_str):
    """
    Convert various date formats to ISO 8601-like string:
      - YYYY-MM-DD if day available
      - YYYY-MM if only month available
      - YYYY if only year available
    Handles ranges (e.g., 08-Jul-2014/11-Oct-2016) by joining with '_'.
    """
    if pd.isna(date_str):
        return 'NA'
    s = str(date_str).strip()

    # Handle date range: split on '/' and standardize each part, then join with '_'
    if '/' in s and not re.match(r'^\d{4}$', s) and not re.match(r'^\d{4}-\d{2}$', s):
        parts = s.split('/')
        std_parts = [standardize_date(p) for p in parts]
        return '_'.join(std_parts)

    # DD-MMM-YYYY (e.g., 13-Jun-2024)
    match = re.match(r'^(\d{1,2})-([A-Za-z]{3})-(\d{4})$', s)
    if match:
        day, mon, year = match.groups()
        month_num = list(calendar.month_abbr).index(mon.capitalize())
        return f"{year}-{month_num:02d}-{int(day):02d}"

    # DD.MM.YYYY (e.g., 02.10.2019)
    match = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{4})$', s)
    if match:
        day, mon, year = match.groups()
        return f"{year}-{int(mon):02d}-{int(day):02d}"

    # YYYY-MM (e.g., 2019-10)
    match = re.match(r'^(\d{4})-(\d{1,2})$', s)
    if match:
        year, mon = match.groups()
        return f"{year}-{int(mon):02d}"

    # MM-YYYY or MM.YYYY (e.g., 11-2025 or 11.2025)
    match = re.match(r'^(\d{1,2})[-.](\d{4})$', s)
    if match:
        mon, year = match.groups()
        return f"{year}-{int(mon):02d}"

    # MMM-YYYY (e.g., Nov-2025)
    match = re.match(r'^([A-Za-z]{3})-(\d{4})$', s)
    if match:
        mon, year = match.groups()
        month_num = list(calendar.month_abbr).index(mon.capitalize())
        return f"{year}-{month_num:02d}"

    # Just YYYY
    match = re.match(r'^(\d{4})$', s)
    if match:
        return s

    # If nothing matches, clean and return as is
    return clean_field(s)


In [7]:
source_qualifiers = ["strain", "isolate", "geo_loc_name", "country", "collection_date", "isolation_source", "host"]
source_qualifiers_columns = ["Strain", "Isolate", "Geo location", "Country", "Collection date", "Tissue/Specimen/Source", "Host"]




meta_path = "D:/Virology/RNA_viruses/Astrovirus/Astroviridae_database/data/results/30-06-2026/final/Astroviridae_30062026_annotated_MAstV_species.tsv"

meta = pd.read_csv(meta_path,sep="\t")
meta = meta.set_index("Accession")
not_in_meta = []
in_meta = []
entries_data = {}
for entry in recs:
    rec_acc = entry.id.split(".")[0]
    if rec_acc not in meta.index:
        not_in_meta.append(rec_acc)
        
        entry_name = entry.id.split(".")[0]
        entries_data[entry_name] = {}
        entries_data[entry_name]['LOCUS'] = entry.name
        entries_data[entry_name]['Version'] = entry.id
        entries_data[entry_name]['GenBank title'] = entry.description
    
        entries_data[entry_name]['Release date'] = entry.annotations['date']
        entries_data[entry_name]['Organism name'] = entry.annotations['organism']
        
        entries_data[entry_name]['Virus Lineage'] = ';'.join(entry.annotations['taxonomy'])
        num_taxa = len(entry.annotations['taxonomy'])
        if num_taxa == 9:
            entries_data[entry_name]['Species'] = entry.annotations['taxonomy'][-1]
        else:
             entries_data[entry_name]['Species'] = 'NA'
        try:
            entries_data[entry_name]['Family'] = entry.annotations['taxonomy'][6]
        except:
            entries_data[entry_name]['Family'] = ""
        
        entries_data[entry_name]['Length'] = len(entry)
    
    
        entries_data[entry_name]['Reference_title'] = ""
        entries_data[entry_name]['Submitters'] = ""
        entries_data[entry_name]['PUBMED_ID'] = ""
        
        if len(entry.annotations['references']) == 1:
            entries_data[entry_name]['Reference_title'] = entry.annotations['references'][0].title
            entries_data[entry_name]['Submitters'] = entry.annotations['references'][0].authors
            entries_data[entry_name]['PUBMED_ID'] = entry.annotations['references'][0].pubmed_id
        else:
            for reference in entry.annotations['references'][:-1]:
                entries_data[entry_name]['Reference_title'] += (reference.title + ';')
                entries_data[entry_name]['Submitters'] += (reference.authors  + ';')
                entries_data[entry_name]['PUBMED_ID'] += (reference.pubmed_id  + ';')
    
            entries_data[entry_name]['Reference_title'] = entries_data[entry_name]['Reference_title'].strip(";")
            entries_data[entry_name]['Submitters'] = entries_data[entry_name]['Submitters'].strip(";")
            entries_data[entry_name]['PUBMED_ID'] = entries_data[entry_name]['PUBMED_ID'].strip(";")
        #for reference in entry.annotations['references']:
        #    if reference.title == 'Direct Submission':
        #        entries_data[entry_name]['Submitters'] = reference.authors
    
        count_cds = 0
        for feature in entry.features:
            if feature.type == 'source':
                for qualif, colname in zip(source_qualifiers, source_qualifiers_columns):
                    if qualif in feature.qualifiers:
                            if qualif == "collection_date":
                                entries_data[entry_name][colname] = standardize_date(feature.qualifiers[qualif][0])
                            else:
                                entries_data[entry_name][colname] = feature.qualifiers[qualif][0]
                    else:
                        entries_data[entry_name][colname] = 'NA'
                if 'environmental_sample' in feature.qualifiers:
                    entries_data[entry_name]['Environmental'] = 'Yes'
                else:
                    entries_data[entry_name]['Environmental'] = 'No'
                    
            if feature.type == 'CDS':
                
                count_cds +=1
                
        entries_data[entry_name]['CDS count'] = count_cds
    
        artificial_condition0 = ('patent' in entry.annotations['references'][-1].journal.lower())
        artificial_condition1 = ("nonfunctional" in entry.description.lower())
    
        if artificial_condition0 or artificial_condition1:
    
            entries_data[entry_name]['Artificial'] = "Yes"
        else:
            entries_data[entry_name]['Artificial'] = "No"
    
    
        if 'UNVERIFIED' in entry.description:
            entries_data[entry_name]['UNVERIFIED'] = "Yes"
        else:
            entries_data[entry_name]['UNVERIFIED'] = "No"
    else:
        in_meta.append(rec_acc)
meta_bastrodataframe = pd.DataFrame.from_dict(entries_data, orient='index')

C:\Users\alxva\AppData\Local\Temp\ipykernel_30368\2926582702.py:9: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(meta_path,sep="\t")


In [10]:
batro_subset = meta.loc[meta.index.intersection(in_meta)]
batro_subset

,LOCUS,Version,GenBank title,Release date,Organism name,Virus Lineage,Species,Family,Length,Reference_title,...,host_clean,Host_species,Host_genus,Host_family,Host_order,Host_class,Host_phylum,Host_kingdom,Species_ICTV,virus_name
Accession,,,,,,,,,,,,,,,,,,,,,
PV605202,PV605202,PV605202.1,MAG: Astroviridae sp. isolate 22AP075 genomic ...,15-OCT-2025,Astroviridae sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5439,Infectome Analysis of Small Mammals in Souther...,...,Suncus murinus,Suncus murinus,Suncus,Soricidae,Eulipotyphla,Mammalia,Chordata,Metazoa,NaN,NaN
PV605168,PV605168,PV605168.1,MAG: Bastrovirus sp. isolate 21DB13 genomic se...,15-OCT-2025,Bastrovirus sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5943,Infectome Analysis of Small Mammals in Souther...,...,Rattus norvegicus,Rattus norvegicus,Rattus,Muridae,Rodentia,Mammalia,Chordata,Metazoa,NaN,NaN
PP872081,PP872081,PP872081.1,MAG: Bastrovirus sp. isolate astro_vt_363 geno...,14-JUN-2025,Bastrovirus sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5359,"Host diversity, viral diversity, and cross-spe...",...,Sus scrofa,Sus scrofa,Sus,Suidae,Artiodactyla,Mammalia,Chordata,Metazoa,NaN,NaN
PP872079,PP872079,PP872079.1,MAG: Bastrovirus sp. isolate astro_vt_361 geno...,14-JUN-2025,Bastrovirus sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5476,"Host diversity, viral diversity, and cross-spe...",...,Sus scrofa domesticus,Sus scrofa,Sus,Suidae,Artiodactyla,Mammalia,Chordata,Metazoa,NaN,NaN
PP872056,PP872056,PP872056.1,MAG: Bastrovirus sp. isolate astro_vt_338 geno...,14-JUN-2025,Bastrovirus sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5476,"Host diversity, viral diversity, and cross-spe...",...,Sus scrofa,Sus scrofa,Sus,Suidae,Artiodactyla,Mammalia,Chordata,Metazoa,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
KU318319,KU318319,KU318319.1,"Bastrovirus 5, partial genome",17-FEB-2016,Bastrovirus 5,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,6040,A novel astrovirus-like RNA virus detected in ...,...,Homo sapiens,Homo sapiens,Homo,Hominidae,Primates,Mammalia,Chordata,Metazoa,NaN,NaN
KU318318,KU318318,KU318318.1,"Bastrovirus 2, partial genome",17-FEB-2016,Bastrovirus 2,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,6087,A novel astrovirus-like RNA virus detected in ...,...,Homo sapiens,Homo sapiens,Homo,Hominidae,Primates,Mammalia,Chordata,Metazoa,NaN,NaN
KU318317,KU318317,KU318317.1,"Bastrovirus 3, complete genome",17-FEB-2016,Bastrovirus 3,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,6336,A novel astrovirus-like RNA virus detected in ...,...,Homo sapiens,Homo sapiens,Homo,Hominidae,Primates,Mammalia,Chordata,Metazoa,NaN,NaN


In [8]:
meta_bastrodataframe

,LOCUS,Version,GenBank title,Release date,Organism name,Virus Lineage,Species,Family,Length,Reference_title,...,Isolate,Geo location,Country,Collection date,Tissue/Specimen/Source,Host,Environmental,CDS count,Artificial,UNVERIFIED
PQ541176,PQ541176,PQ541176.1,MAG: Rodent Bastrovirus isolate Mu. musculus_a...,20-FEB-2025,rodent Bastrovirus,Viruses,NA,,5750,Direct Submission,...,Mu. musculus_astro,Sierra Leone,NA,2023,NA,Mus musculus,Yes,2,No,No
OR710300,OR710300,OR710300.1,Rodent Bastrovirus strain CHN/YN/LH-6/2022 non...,08-NOV-2023,rodent Bastrovirus,Viruses,NA,,5893,Direct Submission,...,DhLh72022,China,NA,2022-07,NA,Rattus tanezumi,No,2,No,No


In [11]:
meta_all = pd.concat([batro_subset,meta_bastrodataframe])

In [12]:
meta_all["Accession"] = meta_all.index

In [13]:
meta_all

,LOCUS,Version,GenBank title,Release date,Organism name,Virus Lineage,Species,Family,Length,Reference_title,...,Host_species,Host_genus,Host_family,Host_order,Host_class,Host_phylum,Host_kingdom,Species_ICTV,virus_name,Accession
PV605202,PV605202,PV605202.1,MAG: Astroviridae sp. isolate 22AP075 genomic ...,15-OCT-2025,Astroviridae sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5439,Infectome Analysis of Small Mammals in Souther...,...,Suncus murinus,Suncus,Soricidae,Eulipotyphla,Mammalia,Chordata,Metazoa,NaN,NaN,PV605202
PV605168,PV605168,PV605168.1,MAG: Bastrovirus sp. isolate 21DB13 genomic se...,15-OCT-2025,Bastrovirus sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5943,Infectome Analysis of Small Mammals in Souther...,...,Rattus norvegicus,Rattus,Muridae,Rodentia,Mammalia,Chordata,Metazoa,NaN,NaN,PV605168
PP872081,PP872081,PP872081.1,MAG: Bastrovirus sp. isolate astro_vt_363 geno...,14-JUN-2025,Bastrovirus sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5359,"Host diversity, viral diversity, and cross-spe...",...,Sus scrofa,Sus,Suidae,Artiodactyla,Mammalia,Chordata,Metazoa,NaN,NaN,PP872081
PP872079,PP872079,PP872079.1,MAG: Bastrovirus sp. isolate astro_vt_361 geno...,14-JUN-2025,Bastrovirus sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5476,"Host diversity, viral diversity, and cross-spe...",...,Sus scrofa,Sus,Suidae,Artiodactyla,Mammalia,Chordata,Metazoa,NaN,NaN,PP872079
PP872056,PP872056,PP872056.1,MAG: Bastrovirus sp. isolate astro_vt_338 geno...,14-JUN-2025,Bastrovirus sp.,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,5476,"Host diversity, viral diversity, and cross-spe...",...,Sus scrofa,Sus,Suidae,Artiodactyla,Mammalia,Chordata,Metazoa,NaN,NaN,PP872056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
KU318317,KU318317,KU318317.1,"Bastrovirus 3, complete genome",17-FEB-2016,Bastrovirus 3,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,6336,A novel astrovirus-like RNA virus detected in ...,...,Homo sapiens,Homo,Hominidae,Primates,Mammalia,Chordata,Metazoa,NaN,NaN,KU318317
KU318316,KU318316,KU318316.1,"Bastrovirus 6, partial genome",17-FEB-2016,Bastrovirus 6,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,6017,A novel astrovirus-like RNA virus detected in ...,...,Homo sapiens,Homo,Hominidae,Primates,Mammalia,Chordata,Metazoa,NaN,NaN,KU318316
KU318315,KU318315,KU318315.1,"Bastrovirus 1, partial genome",17-FEB-2016,Bastrovirus 1,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,6207,A novel astrovirus-like RNA virus detected in ...,...,Homo sapiens,Homo,Hominidae,Primates,Mammalia,Chordata,Metazoa,NaN,NaN,KU318315
PQ541176,PQ541176,PQ541176.1,MAG: Rodent Bastrovirus isolate Mu. musculus_a...,20-FEB-2025,rodent Bastrovirus,Viruses,NA,,5750,Direct Submission,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PQ541176


In [14]:
meta_all.to_csv("meta.tsv",sep="\t")

Get coding sequences for future hmmsearch

In [19]:
#======= Analyses CDS feature which annotation does not correspond to target CDS =======
def analyze_cds(record, feature, transl_table=1):
    """
    Extracts translation and coordinates from a CDS feature.
    Returns a dict with details, including internal stop count.
    """
    
    codon_start = int(feature.qualifiers.get("codon_start", [1])[0])
    offset = codon_start - 1


    # Get parts – if not a compound location, treat as single part
    if len(feature.location.parts) > 1:
        parts = feature.location.parts
    else:
        parts = [feature.location]

    
    candidates = []
    for idx, part in enumerate(parts):
        raw_seq = part.extract(record.seq)
        # codon_start offset only applies to the first part
        if idx == 0:
            cds_seq = raw_seq[offset:]
        else:
            cds_seq = raw_seq

        # Translate, include stop codon
        translation = cds_seq.translate(table=transl_table, to_stop=False)
        annotated_translation = feature.qualifiers.get("translation", [""])[0]
        
        # Count internal stops (* not at last position)
        stops = [i for i, aa in enumerate(translation) if aa == "*" and i != len(translation)-1]
        internal_stop_count = len(stops)
        
        if translation and translation[-1] == "*":
            translation = translation[:-1]

        if len(parts) == 1 and translation != annotated_translation:
            print(f"For {record.id} the annotated translation does not correspond to actual translation of coding sequence")
            print("Actual translation: " + translation)
            print("Annotated translation: " + annotated_translation)


        start = int(part.start)
        end = int(part.end)
        
        cand = {
            "record_name": record.name,
            "accession": record.id,
            "location": str(part),                # individual part location
            "codon_start": str(codon_start if idx == 0 else 1),
            "start": start,
            "end": end,
            "strand": part.strand,
            "gene": feature.qualifiers.get("gene", ["unknown"])[0],
            "product": feature.qualifiers.get("product", ["unknown"])[0],
            "transl_table": transl_table,
            "raw_nucleotide": str(raw_seq),
            "cds_sequence": str(cds_seq),
            "translation": str(translation),
            "translation_annot": str(annotated_translation),
            "internal_stop_count": internal_stop_count,
            "is_clean": internal_stop_count == 0,
            "part_index": idx,   # optional, for debugging
        }
        candidates.append(cand)
    return candidates

#======= Validation and correction of frame in CDS feature which annotation does not correspond to target CDS =======
def correct_cds_frame(cds_info, max_stops=2):
    """
    Attempt to correct the translation frame of a CDS.
    Tries all 3 reading frames on the raw nucleotide sequence.
    Returns the best translation and the offset to apply.

    Parameters:
    cds_info - dictionary with information of CDS feature, output of analyze_cds funciton
    max_stops - maximal number of stop codons
    """
    raw = cds_info["raw_nucleotide"]  # use the untrimmed raw sequence
    transl_table = cds_info["transl_table"]
    
    best = None
    best_offset = None
    
    for frame in range(3):  # 0, 1, 2
        seq = Seq(raw[frame:])
        
        trans = seq.translate(table=transl_table, to_stop=False)
        # count internal stops
        internal_stops = [i for i, aa in enumerate(trans) if aa == "*" and i != len(trans)-1]
        stop_count = len(internal_stops)
        
        # We prefer fewer stops, and if equal, longer length
        if stop_count <= max_stops:
            if best is None or (stop_count < best["stop_count"]) or \
               (stop_count == best["stop_count"] and len(trans) > len(best["translation"])):
                best = {
                    "translation": str(trans),
                    "stop_count": stop_count,
                    "frame_offset": frame,  # number of bases to skip from the raw start
                    "nucleotide_used": str(seq),
                }
                best_offset = frame
    return best, best_offset
# ======= write CDS with no target or conflicting annotation
def write_fasta_and_mapping(candidates, fasta_file, mapping_file):
    """
    Writes two files:
      - fasta_file: amino acid sequences with header containing accession and some metadata.
      - mapping_file: TSV (or CSV) with detailed coordinates/strand per candidate.
    """
    with open(fasta_file, "w") as f_fasta, open(mapping_file, "w", newline="") as f_map:
        writer = csv.writer(f_map, delimiter="\t")
        writer.writerow(["candidate_id",
            "record_name", "accession", "start", "end", "codon_start","strand",
            "gene", "product", "is_corrected", "correction_offset", "original_translation"
        ])
        for idx, c in enumerate(candidates, 1):
            # FASTA header: include ID and some quick info
            cid = f"cand_{idx:04d}"
            header = f">{cid}|{c['accession']}|{c['gene']}|start={c['start']}|end={c['end']}|codon_start={c['codon_start']}|strand={c['strand']}"
            f_fasta.write(header + "\n")
            trans = c['translation']
            for i in range(0, len(trans), 60):
                f_fasta.write(trans[i:i+60] + "\n")
            
            # Write mapping line
            writer.writerow([
                cid,
                c['record_name'],
                c['accession'],
                c['start'],
                c['end'],
                c['codon_start'],
                c['strand'],
                c['gene'],
                c['product'],
                c.get('is_corrected', False),
                c.get('correction_frame_offset', ''),
                c.get('original_translation', '')
            ])

In [20]:
len(recs_noamb)

67

In [21]:
cds_candidates = []
cds_counts = []
for rec in recs_noamb:
    cds_count = 0
    
    rec_name = rec.id.split(".")[0]

    for feature in rec.features:
        if feature.type == 'CDS':
            cds_count +=1
            #print(feature)
            cds_infos = analyze_cds(rec, feature)
            for cds_info in cds_infos:
                if not cds_info["is_clean"]:
                    corrected, offset = correct_cds_frame(cds_info, max_stops=1)
                    if corrected is not None:
                        cds_info["is_corrected"] = True
                        cds_info["corrected_translation"] = corrected["translation"]
                        cds_info["original_translation"] = cds_info["translation"]
                        cds_info["translation"] = corrected["translation"]
                        cds_info["correction_frame_offset"] = offset
                        cds_info["codon_start"] = int(cds_info["codon_start"]) + offset
                    else:
                        cds_info["is_corrected"] = False
                        cds_info["correction_failed"] = True
                cds_candidates.append(cds_info)
    print(f"{rec.name} has {cds_count} annotated CDS")
    cds_counts.append(cds_count)
    print("_______")

out_folder = "cds"
out_file_name_temp = "bastro"
#os.mkdir(out_folder)
# Write all CDS

fasta_cand = os.path.join(out_folder, out_file_name_temp + '_cds.faa')
mapping_cand = os.path.join(out_folder, out_file_name_temp + '_cds.tsv')
write_fasta_and_mapping(cds_candidates, fasta_cand, mapping_cand)

For PV605168.1 the annotated translation does not correspond to actual translation of coding sequence
Actual translation: LDAVNGRGHTPKQLLWLQNPDVNHFQAVAVEGLLPLLPCDVIGSVDTNAFVGTLPAVLHAVNAVPGQVGALINHGPLKPQLIACVVHETAHVHCPPLDVAEVTTNVNRALTMPVSMVVSEGVQCAHHGMRVCDAILEHEVAPQNLEEVFLLLAVEHERNVQGIADSACHCTAEACAVLTFPVAALGVREGGGAGERSM
Annotated translation: MDAVNGRGHTPKQLLWLQNPDVNHFQAVAVEGLLPLLPCDVIGSVDTNAFVGTLPAVLHAVNAVPGQVGALINHGPLKPQLIACVVHETAHVHCPPLDVAEVTTNVNRALTMPVSMVVSEGVQCAHHGMRVCDAILEHEVAPQNLEEVFLLLAVEHERNVQGIADSACHCTAEACAVLTFPVAALGVREGGGAGERSM
For PV605168.1 the annotated translation does not correspond to actual translation of coding sequence
Actual translation: LHIPFVLNREEKEYLLKVLRCDLVFEDRVAHTHSMVRALHAFANHHAYRHCEGAIDIGGDFRNIKRRAVHVCGLVDNARDELRLQRAMVDQRPNLAGYCIHGVENCRERADKGICVHAAYDITWQQWQQAFDSHGLKVVDVWILEPEELFGRVTPPIYGIQMHRLANRWAMLLDDACAGYVHDDFEWRKYRETGGTIGPRFNIGCSPEGGWGPLRRYTLFRCVHPFRQITLRHPVDRNTVHVPTPAGHVISCPRSVWQNLLSWACSRADNKFDFSSLLAYARALRQRLTIGGQTVQEGWSQDPRQVHELCIACFVMAAAMRYTRTQNIAVAFAEL

C:\Python310\lib\site-packages\Bio\Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


Scan obtained CDS against PFAM

In [22]:
import copy
def load_domain_map(csv_file):
    """
    Load domain-to-ORF mapping from a CSV file.
    Expected columns: domain_name, orf_type
    Returns a dict {domain_name: orf_type}.
    """
    domain_map = {}
    with open(csv_file, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            domain = row['domain_name'].strip()
            orf = row['orf_type'].strip()
            if domain and orf:
                domain_map[domain] = orf
    return domain_map

def parse_hmmscan_domtbl(domtbl_file, evalue_threshold=0.01):
    """
    Parse hmmscan --domtblout output.
    Returns a dict: {query_id: [list_of_domain_hits]}
    Each hit is a dict with keys:
        'domain_name', 'domain_accession', 'domain_description',
        'query_start', 'query_end', 'score', 'evalue', 'bitscore'
    """
    domain_hits = {}
    with open(domtbl_file) as f:
        for line in f:
            if line.startswith('#'):
                continue
            fields = line.strip().split()
            if len(fields) < 23:
                continue
            # Fields (0‑based indices):
            # 0: target name (domain)
            # 1: target accession
            # 2: tlen
            # 3: query name (your sequence ID)
            # 4: query accession (often '-')
            # 5: qlen
            # 6: evalue (full sequence)
            # 7: score (full sequence)
            # 8: bias
            # 9: domain number
            # 10: domain count in this hit
            # 11: domain c-Evalue
            # 12: domain i-Evalue
            # 13: domain score
            # 14: domain bias
            # 15: hmm coord from
            # 16: hmm coord to
            # 17: ali coord from (query start)
            # 18: ali coord to (query end)
            # 19: env coord from
            # 20: env coord to
            # 21: acc (optional)
            # 22: description of target (the rest)
            query = fields[3]
            domain_name = fields[0]
            domain_acc = fields[1]
            domain_evalue = float(fields[11])
            if domain_evalue > evalue_threshold:
                continue
            query_start = int(fields[17])
            query_end = int(fields[18])
            bitscore = float(fields[13])
            # Full sequence description (may contain spaces)
            desc = ' '.join(fields[22:]) if len(fields) > 22 else ''

            
            hit = {
                'domain_name': domain_name,
                'domain_accession': domain_acc,
                'domain_description': desc,
                'query_start': query_start,
                'query_end': query_end,
                'score': bitscore,
                'evalue': domain_evalue
            }
            domain_hits.setdefault(query, []).append(hit)
    return domain_hits

def assign_orf_from_domains_old(domain_hits, domain_map, min_score=0, min_coverage=0.5):
    """
    Given a list of domain hits for a single sequence,
    return the most likely ORF type based on the best hit (lowest E-value).
    If no hit matches the map, return None.
    """
    if not domain_hits:
        return None
    # Sort by E-value (ascending)
    sorted_hits = sorted(domain_hits, key=lambda x: x['evalue'])
    domains = []
    coords = {}
    for hit in sorted_hits:
        dom_name = hit['domain_name']
        if dom_name in domain_map:
            domains.append(domain_map[dom_name])
            if domain_map[dom_name] == "1B" or domain_map[dom_name] == "MetTransf":
                #print(hit)
                coords[domain_map[dom_name]] = [hit["query_start"],hit["query_end"]]
            
                
    domains = list(set(domains))
    if len(domains) > 1:
        print("Two domains are found in one CDS")
        print(domains)
        return coords
    if len(domains) == 1:
        if domains[0] == '1B':
            coords['1B'] = [hit["query_start"],hit["query_end"]]
            return coords
        else:
            return domains[0]
    else:
        return None

def assign_problematic_orfs_old(candidate_tsv, domain_table_file, domain_map_file, output_tsv):
    """
    For each problematic candidate, use the domain hits to assign an ORF type.
    Writes a new TSV with an additional 'assigned_orf' column.
    """
    domain_map = load_domain_map(domain_map_file)
    domain_hits = parse_hmmscan_domtbl(domain_table_file)
    with open(candidate_tsv) as f:
        reader = csv.DictReader(f, delimiter='\t')
        rows = list(reader)

    not_assigned_rows = []
    assigned_rows = []
    for row in rows:
        # Reconstruct the FASTA header from the candidate info
        cid = row['candidate_id']
        header = f"{cid}|{row['accession']}|{row['gene']}|start={row['start']}|end={row['end']}|codon_start={row['codon_start']}|strand={row['strand']}"
        hits = domain_hits.get(header, [])
        orf_type = assign_orf_from_domains(hits, domain_map)
        if type(orf_type) is dict:
            for orf,coords in orf_type.items():
                row1 = copy.deepcopy(row)
                row1['assigned_orf'] = orf
                row1['dom_start'] = coords[0]
                row1['dom_end'] = coords[1]
                assigned_rows.append(row1)
        else:    
            row['assigned_orf'] = orf_type if orf_type else 'unknown'
            assigned_rows.append(row)
        if not orf_type:
            not_assigned_rows.append(row)
        
    print(f"{len(not_assigned_rows)} CDS were not assigned")
    
    # Write updated TSV
    with open(output_tsv, 'w', newline='') as f:
        fieldnames = reader.fieldnames + ['assigned_orf', 'dom_start', 'dom_end']
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter='\t')
        writer.writeheader()
        writer.writerows(assigned_rows)

def assign_orf_from_domains(domain_hits, domain_map, target_orf,
                            min_evalue=0.00001, min_coverage=0.0, query_len=None,
                            merge_gap=0, mode="merged"):
    """
    Extract the region covered by hits whose domain_name maps to `target_orf`.

    Parameters
    ----------
    domain_hits : list of dict
        Output of parse_hmmscan_domtbl for one query.
    domain_map : dict
        {domain_name: orf_type}
    target_orf : str
        "1B" for RdRp, "2" for capsid, etc.
    min_evalue : float
        Minimum bitscore for a hit to be kept.
    min_coverage : float
        Minimum fraction of query covered by the returned region.
        Ignored if query_len is None.
    query_len : int or None
        Length of the query protein (for coverage check).
    merge_gap : int
        Merge intervals separated by at most this many residues.
    mode : {"merged", "envelope"}
        "merged"   -> pick the widest merged contiguous block (recommended).
        "envelope" -> min(start), max(end) over all kept hits.

    Returns
    -------
    dict or None
        {
          'orf_type': target_orf,
          'region': (start, end),          # 1-based inclusive
          'merged_intervals': [(s,e),...],
          'hits': [...],
          'n_hits': int,
        }
    """
    if not domain_hits:
        return None
    selected = [
        h for h in domain_hits
        if domain_map.get(h['domain_name']) == target_orf
        and h.get('evalue', 1) <= min_evalue
    ]
    if not selected:
        return None
    intervals = sorted(
        (int(h['query_start']), int(h['query_end'])) for h in selected
    )

    # merge overlapping / book-ended / near-adjacent intervals
    merged = []
    cur_s, cur_e = intervals[0]
    for s, e in intervals[1:]:
        if s <= cur_e + 1 + merge_gap:
            cur_e = max(cur_e, e)
        else:
            merged.append((cur_s, cur_e))
            cur_s, cur_e = s, e
    merged.append((cur_s, cur_e))

    if mode == "envelope":
        region = (min(s for s, _ in intervals), max(e for _, e in intervals))
    else:  # "merged" -> widest single merged block
        region = max(merged, key=lambda iv: iv[1] - iv[0])

    if query_len is not None and min_coverage > 0:
        span = region[1] - region[0] + 1
        if span / query_len < min_coverage:
            return None

    return {
        'orf_type': target_orf,
        'region': region,
        'merged_intervals': merged,
        'hits': selected,
        'n_hits': len(selected),
    }
    

def has_orf_support(domain_hits, domain_map, target_orf,
                    min_evalue=1, required_domains=None):
    """
    Return True if at least one hit maps to target_orf and passes min_evalue.
    If required_domains is given, ALL of them must be present among the hits.
    """
    present = {
        h['domain_name'] for h in domain_hits
        if domain_map.get(h['domain_name']) == target_orf
        and h.get('evalue', 1) <= min_evalue
    }

    if not present:
        return False
    if required_domains and not set(required_domains).issubset(present):
        return False
    return True

def assign_problematic_orfs(candidate_tsv, domain_table_file, domain_map_file,
                            output_tsv,
                            targets=None,
                            min_evalue=0.00001,
                            min_coverage=0.4,
                            merge_gap_by_target=None):
    """
    For each problematic candidate, run domain-based ORF assignment for each
    target ORF type separately, and write one output row per (candidate, target)
    where a region was found. Candidates with no region for any target get a
    single 'unknown' row.

    Parameters
    ----------
    targets : list of (orf_type, merge_gap, required_domains or None)
        e.g. [("1B", 5, None), ("2", 45, None)]
    min_evalue, min_coverage : filters passed to assign_orf_from_domains
    merge_gap_by_target : optional dict {orf_type: gap}; overrides the gap in `targets`
    """
    if targets is None:
        targets = [
            ("1B", 5,  None),                # RdRp: widest merged interval
            ("2",  0, None),  # Capsid: whole ORF if any capsid hit (gap unused)
            ("CaliciCapsid",  0, None),
            ("MetTransf",  5, None)
        ]
    if merge_gap_by_target:
        targets = [(t, merge_gap_by_target.get(t, g), r) for t, g, r in targets]

    domain_map    = load_domain_map(domain_map_file)
    domain_hits   = parse_hmmscan_domtbl(domain_table_file)

    with open(candidate_tsv) as f:
        reader = csv.DictReader(f, delimiter='\t')
        rows = list(reader)
        in_fieldnames = reader.fieldnames

    assigned_rows    = []
    not_assigned_rows = []

    for row in rows:
        cid    = row['candidate_id']
        header = (f"{cid}|{row['accession']}|{row['gene']}"
                  f"|start={row['start']}|end={row['end']}"
                  f"|codon_start={row['codon_start']}|strand={row['strand']}")

        # hmmscan stores the FASTA header verbatim up to whitespace; try both keys
        hits = domain_hits.get(header) or domain_hits.get(cid) or []
        if cid == "cand_0043":
            print(hits)
        # query_len from candidate row, used for the coverage check
        try:
            qlen = int(row.get('aa_len')) if row.get('aa_len') else None
        except ValueError:
            qlen = None
        if qlen is None:
            # fall back: nucleotide CDS length / 3 if you have it
            try:
                qlen = len(row['cds_sequence']) // 3 if row.get('cds_sequence') else None
            except Exception:
                qlen = None

        found_any = False
        for orf_type, gap, required in targets:
            if orf_type == "2" or orf_type == "CaliciCapsid":
                # whole-ORF assignment: no interval logic
                if not has_orf_support(hits, domain_map, orf_type,
                                       min_evalue=min_evalue,
                                       required_domains=required):
                    continue
                out = copy.deepcopy(row)
                out['assigned_orf']     = orf_type
                out['dom_start']        = 1
                out['dom_end']          = qlen if qlen else ''
                out['n_hits']           = sum(
                    1 for h in hits
                    if domain_map.get(h['domain_name']) == "2"
                    and h.get('evalue', 1) <= min_evalue
                )
                out['merged_intervals'] = ''
                assigned_rows.append(out)
                found_any = True
            else:
                res = assign_orf_from_domains(
                    hits, domain_map,
                    target_orf       = orf_type,
                    min_evalue        = min_evalue,
                    min_coverage     = min_coverage,
                    query_len        = qlen,
                    merge_gap        = gap,
                    mode             = "merged"
                )
                if not res:
                    continue
                found_any = True
                out = copy.deepcopy(row)
                out['assigned_orf'] = orf_type
                out['dom_start']    = res['region'][0]
                out['dom_end']      = res['region'][1]
                out['n_hits']       = res['n_hits']
                out['merged_intervals'] = ";".join(f"{s}-{e}"
                                                   for s, e in res['merged_intervals'])
                assigned_rows.append(out)

        if not found_any:
            out = copy.deepcopy(row)
            out['assigned_orf'] = 'unknown'
            out['dom_start']    = ''
            out['dom_end']      = ''
            out['n_hits']       = 0
            out['merged_intervals'] = ''
            assigned_rows.append(out)
            not_assigned_rows.append(out)

    print(f"{len(not_assigned_rows)} CDS were not assigned")

    fieldnames = in_fieldnames + [
        'assigned_orf', 'dom_start', 'dom_end',
        'n_hits', 'merged_intervals',
    ]
    with open(output_tsv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter='\t')
        writer.writeheader()
        writer.writerows(assigned_rows)

    return assigned_rows, not_assigned_rows



In [24]:
assign_problematic_orfs("cds/bastro_cds.tsv", "hmmer/bastro_cds_domains.tbl", "hmmer/domain_map.csv", "hmmer/bastro_cds_assigned.tsv")

[{'domain_name': 'Calici_coat', 'domain_accession': 'PF00915.27', 'domain_description': 'Calicivirus coat protein', 'query_start': 69, 'query_end': 133, 'score': 18.1, 'evalue': 5.5e-08}]
22 CDS were not assigned


([{'candidate_id': 'cand_0001',
   'record_name': 'PV605168',
   'accession': 'PV605168.1',
   'start': '0',
   'end': '595',
   'codon_start': '1',
   'strand': '-1',
   'gene': 'ORF3',
   'product': 'hypothetical protein',
   'is_corrected': 'False',
   'correction_offset': '',
   'original_translation': '',
   'assigned_orf': 'unknown',
   'dom_start': '',
   'dom_end': '',
   'n_hits': 0,
   'merged_intervals': ''},
  {'candidate_id': 'cand_0002',
   'record_name': 'PV605168',
   'accession': 'PV605168.1',
   'start': '112',
   'end': '3688',
   'codon_start': '1',
   'strand': '1',
   'gene': 'ORF1',
   'product': 'putative non-structural polyprotein',
   'is_corrected': 'False',
   'correction_offset': '',
   'original_translation': '',
   'assigned_orf': '1B',
   'dom_start': 804,
   'dom_end': 1171,
   'n_hits': 1,
   'merged_intervals': '804-1171'},
  {'candidate_id': 'cand_0002',
   'record_name': 'PV605168',
   'accession': 'PV605168.1',
   'start': '112',
   'end': '3688',


## Retrieve protein sequences of assigned ORFs

In [25]:
records_cds = SeqIO.parse("cds/bastro_cds.faa", "fasta")
assigned_tsv = pd.read_csv("hmmer/bastro_cds_assigned.tsv", sep="\t") 
meta = pd.read_csv("meta.tsv",sep="\t")

meta = meta.set_index("Accession")
#assigned_tsv = assigned_tsv.set_index("record_name")


recs_RdRp = []
recs_capsid = []

for rec in records_cds:
    recid = rec.id.split("|")[1].split(".")[0]
    cid = rec.id.split("|")[0]
    organism = meta.loc[recid]["Organism name"].replace(" ", "-")
    host = meta.loc[recid]["host_clean"]
    if type(host) == str:
        host = host.replace(" ", "-")
    else:
        host = "NA"
    rec.name = recid + "|" + organism + "|" +  host + "|" + cid
    rec.id = rec.name
    rec.description=""
    
    
    cand_tsv = assigned_tsv[assigned_tsv["candidate_id"] == cid]
    print("______________")
    print(cand_tsv)
    if len(cand_tsv) == 1:
        domain = cand_tsv["assigned_orf"].iloc[0]
        if domain == "1B":
            st = int(cand_tsv["dom_start"].iloc[0])
            e = int(cand_tsv["dom_end"].iloc[0])
            if e < st:
                st1 = st
                st = e
                e = st1
            rec.seq = rec.seq[st-1:e]
            
            recs_RdRp.append(rec)
            print(f"For {rec.id} appended ORF1b with coordinates {st}-{e}")
        elif domain == "2" or domain == "CaliciCapsid":
            recs_capsid.append(rec)
            print(f"For {rec.id} appended capsid")
    else:
        print("Several proteins")
        for row in cand_tsv.iterrows():
            domain = row[1]["assigned_orf"]
            print(domain)
            if domain == "1B":
                st = int(row[1]["dom_start"])
                e = int(row[1]["dom_end"])
                if e < st:
                    st1 = st
                    st = e
                    e = st1
                rec.seq = rec.seq[st-1:e]
                recs_RdRp.append(rec)
                print(f"For {rec.id} appended ORF1b with coordinates {st}-{e}")
            elif domain == "2" or domain == "CaliciCapsid":
                recs_capsid.append(rec)
                print(f"For {rec.id} appended capsid")

SeqIO.write(recs_RdRp, "cds_assigned/bastro_rdrp.fasta", "fasta")
SeqIO.write(recs_capsid, "cds_assigned/bastro_capsid.fasta", "fasta")

______________
  candidate_id record_name   accession  start  end  codon_start  strand  gene  \
0    cand_0001    PV605168  PV605168.1      0  595            1      -1  ORF3   

                product  is_corrected  correction_offset  \
0  hypothetical protein         False                NaN   

   original_translation assigned_orf  dom_start  dom_end  n_hits  \
0                   NaN      unknown        NaN      NaN       0   

  merged_intervals  
0              NaN  
______________
  candidate_id record_name   accession  start   end  codon_start  strand  \
1    cand_0002    PV605168  PV605168.1    112  3688            1       1   
2    cand_0002    PV605168  PV605168.1    112  3688            1       1   

   gene                              product  is_corrected  correction_offset  \
1  ORF1  putative non-structural polyprotein         False                NaN   
2  ORF1  putative non-structural polyprotein         False                NaN   

   original_translation assigned_o

61

# Remove sequence with ambiguous aa in RdRp or capsid

In [26]:
fasta_rdrp = "cds_assigned/bastro_rdrp.fasta"
fasta_capsid = "cds_assigned/bastro_capsid.fasta"
path = "cds_assigned/"
recs_noamb = {}

for protein in ["rdrp", "capsid"]:
    recs_noamb[protein] = []
    recs = list(SeqIO.parse(f"{path}bastro_{protein}.fasta", "fasta"))
    for rec in recs:
        count_X = rec.seq.count("X")
        if count_X == 0:
            recs_noamb[protein].append(rec.id.split('|')[0])
    print(f"{len(recs_noamb[protein])} sequences with no amb aa for {protein}")

65 sequences with no amb aa for rdrp
58 sequences with no amb aa for capsid


In [27]:
intersection = list(set(recs_noamb['rdrp']) & set(recs_noamb['capsid']))
print(f"{len(intersection)} sequences intersect")

56 sequences intersect


In [28]:
for protein in ["rdrp", "capsid"]:
    recs = list(SeqIO.parse(f"{path}bastro_{protein}.fasta", "fasta"))
    recs_intersect =  []
    for rec in recs:
        rec_id = rec.id.split('|')[0]
        if rec_id  in intersection:
            recs_intersect.append(rec)
    SeqIO.write(recs_intersect, f"{path}bastro_{protein}_noamb.fasta", "fasta")

# Remove similar sequences

In [39]:
mmseqs_path = f"orf1b_mmseq_clustering"
mmseq_results = pd.read_table(os.path.join(mmseqs_path, "bastro_1B_aa70_mmseq2.tsv"), header=None)
len(mmseq_results[0].unique())


rdrp_path = f"{path}bastro_rdrp_noamb.fasta"
output_rdrp_path = f"{path}bastro_rdrp_noamb_reduced.fasta"

cluster_centers = mmseq_results[0].unique()
cluster_centers = [x.split("|")[0] for x in cluster_centers]

with open(rdrp_path, "r") as input_file:
    seq_dict = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))
input_file.close()


recs_clustered = []
for rec_name, rec in seq_dict.items():
    rec_id = rec.id.split("|")[0]
    if rec_id in cluster_centers:
        recs_clustered.append(rec)
SeqIO.write(recs_clustered, output_rdrp_path, "fasta")


capsid_recs = SeqIO.parse(f"{path}bastro_capsid_noamb.fasta", "fasta")
capsid_recs_reduced = []
for rec in capsid_recs:
    rec_id = rec.id.split("|")[0]
    if rec_id in cluster_centers:
        capsid_recs_reduced.append(rec)
SeqIO.write(capsid_recs_reduced, f"{path}bastro_capsid_noamb_reduced.fasta", "fasta")

11